# Integração SINAN–IBGE/SIDRA

Município de residência (`ID_MN_RESI`), população residente do mesmo ano e saneamento somente nos anos censitários compatíveis. Classificação final histórica: `CON_CLASSI`, nunca o antecedente `DENGUE`.

**Período anual:** `ano` nos contratos SINAN identifica o ano-fonte (`periodo_referencia = ano_fonte_sinan`), não necessariamente um ano civil completo. Datas em anos adjacentes são preservadas e auditadas. A população usa o ano de referência; as séries diárias climáticas permanecem separadas, pela data efetiva de notificação.

A coorte original inclui uma notificação de 2008 com data corrompida. A execução usa `allow_undated=True`: o registro permanece no período-fonte anual e na auditoria, sem inventar dia ou mês; gráficos por data não o incluem.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "utils/analysis_pipeline.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import polars as pl
from IPython.display import display

import plotly.express as px
from utils.analysis_pipeline import build_integrated_outputs
dados = build_integrated_outputs(allow_undated=True)
display(dados["auditoria"])
display(dados["qualidade"])

/Users/thitav/Documents/Pysus-IBGE/utils/analysis_pipeline.py:410: UserWarning: 1 registro(s) sem data válida mantido(s) apenas pelo ano-fonte; a exportação mensal conserva ano/mês civil desconhecido. Consulte auditoria_sinan_datas_invalidas.csv.
  warnings.warn(


base,ano,linhas_esperadas,linhas_totais,linhas_elegiveis,chaves_unicas,grade_censo_completa,populacao_valida,correspondencias,percentual_correspondencia,municipios_com_notificacao,municipios_sem_notificacao,percentual_cobertura_sinan,notificacoes,casos_confirmados,registros_sinan_brutos,registros_codigo_residencia_invalido,registros_codigo_notificacao_invalido,registros_residencia_notificacao_divergente,municipios_sinan_sem_correspondencia_populacao,notificacoes_sem_correspondencia_populacao
str,i64,i64,i64,i64,i64,bool,i64,i64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64
"""populacao_2010""",2010,5565,5565,5565,5565,true,5565,5565,100.0,4572,993,82.156334,1381060,869169,1381254,127,0,59765,31,67
"""saneamento_2010""",2010,5565,5565,5565,5565,true,5565,5565,100.0,4572,993,82.156334,1381060,869169,1381254,127,0,59765,31,67
"""populacao_2022""",2022,5570,5570,5570,5570,true,5570,5570,100.0,5017,553,90.071813,1399370,1250896,1405095,193,0,75659,37,5532
"""saneamento_2022""",2022,5570,5570,5570,5570,true,5570,5570,100.0,5017,553,90.071813,1399370,1250896,1405095,193,0,75659,37,5532


arquivo,ano_arquivo,registros,codigo_municipio_residencia_invalido,codigo_municipio_notificacao_invalido,codigo_municipio_invalido,municipio_residencia_notificacao_divergente,data_invalida,ano_informado_divergente,ano_data_divergente
str,i32,u32,u32,u32,u32,u32,u32,u32,u32
"""DENGBR00.parquet""",2000,172855,1,0,1,7207,0,48,0
"""DENGBR01.parquet""",2001,488590,34,0,34,14012,0,243,0
"""DENGBR02.parquet""",2002,897093,255,0,255,42574,0,202,0
"""DENGBR03.parquet""",2003,416609,311,0,311,16742,0,20,0
"""DENGBR04.parquet""",2004,136867,67,0,67,5507,0,0,0
…,…,…,…,…,…,…,…,…,…
"""DENGBR19.parquet""",2019,2261956,52,0,52,138127,0,1809,1809
"""DENGBR20.parquet""",2020,1495117,35,0,35,69046,0,6800,6800
"""DENGBR21.parquet""",2021,1024635,77,0,77,55955,0,1466,1466


In [2]:
resumo = []
for year, key in [(2010, "saneamento_integrado"), (2022, "saneamento_2022_integrado")]:
    frame = dados[key]
    resumo.append({"ano":year, "municipios_grade":frame.height,
                   "municipios_com_notificacao":int(frame["tem_notificacao_registrada"].sum()),
                   "municipios_com_sidra":int(frame["correspondencia_sidra"].sum()),
                   "notificacoes":int(frame["notificacoes"].sum()),
                   "casos_confirmados":int(frame["casos_confirmados"].sum())})
display(pl.DataFrame(resumo))

ano,municipios_grade,municipios_com_notificacao,municipios_com_sidra,notificacoes,casos_confirmados
i64,i64,i64,i64,i64,i64
2010,5565,4572,5565,1381060,869169
2022,5570,5017,5570,1399370,1250896


In [3]:
classificacao = (
    dados['classificacao']
    .group_by('status_classificacao')
    .agg(pl.col('registros').sum())
    .with_columns(
        (pl.col('registros') / pl.col('registros').sum() * 100).alias('percentual')
    )
    .sort('registros', descending=True)
)
display(classificacao)
fig = px.bar(
    classificacao.to_pandas(), x='status_classificacao', y='registros',
    text_auto='.3s', title='Classificação das notificações SINAN — anos-fonte disponíveis',
)
fig.show()

status_classificacao,registros,percentual
str,u32,f64
"""confirmado""",14280945,57.053032
"""descartado""",7216926,28.831951
"""inconclusivo""",2913626,11.64007
"""nao_preenchido""",464940,1.857457
"""outro_codigo""",154564,0.61749


In [4]:
painel = dados["populacao_integrada"]
serie_anual = (
    painel
    .group_by('ano')
    .agg(
        pl.col('notificacoes').sum(),
        pl.col('casos_confirmados').sum(),
        pl.col('codigo_municipio').n_unique().alias('municipios'),
    )
    .sort('ano')
)
display(serie_anual)
fig = px.line(
    serie_anual.to_pandas(), x='ano', y=['notificacoes', 'casos_confirmados'],
    markers=True, title='Notificações e casos confirmados por ano-fonte',
)
fig.show()

ano,notificacoes,casos_confirmados,municipios
i32,u64,u64,u32
2000,172854,99349,2101
2001,488556,286610,5593
2002,896838,481260,5602
2003,416298,227424,5592
2004,136800,63483,5583
…,…,…,…
2019,2261904,1311737,5580
2020,1495082,793006,5595
2021,1024558,455711,5592


In [5]:
serie_mensal = dados["mensal"].filter(pl.col("mes").is_not_null()).group_by("ano", "mes").agg(pl.col("notificacoes").sum()).sort("ano", "mes").to_pandas()
fig = px.line(
    serie_mensal, x='mes', y='notificacoes', color='ano',
    title='Sazonalidade mensal das notificações por ano-fonte (mês da notificação)',
).update_xaxes(dtick=1)
fig.show()

## Associação descritiva em 2022

Mesma amostra municipal completa para todos os indicadores. São correlações ecológicas sem interpretação causal, sujeitas a confundimento e dependência espacial. O notebook espacial fornece sensibilidade à população e inferência local com FDR-BY.

In [6]:
indicators = ["pct_agua_rede_geral_principal", "pct_esgoto_rede_geral_ou_pluvial", "pct_sem_banheiro_ou_sanitario", "pct_lixo_queimado_propriedade", "pct_lixo_terreno_encosta_area_publica"]
complete = dados["saneamento_2022_integrado"].drop_nulls([*indicators, "taxa_confirmados_100k"])
display(pl.DataFrame({"indicador":indicators, "n_comum":[complete.height] * len(indicators),
                     "r_pearson":[complete.select(pl.corr(c, "taxa_confirmados_100k")).item() for c in indicators]}))
px.scatter(complete.to_pandas(), x="pct_esgoto_rede_geral_ou_pluvial", y="taxa_confirmados_100k", hover_name="municipio", opacity=0.45, title="Esgotamento e taxa de casos confirmados — municípios, 2022").show()

indicador,n_comum,r_pearson
str,i64,f64
"""pct_agua_rede_geral_principal""",5570,0.162065
"""pct_esgoto_rede_geral_ou_pluvi…",5570,0.12224
"""pct_sem_banheiro_ou_sanitario""",5570,-0.105768
"""pct_lixo_queimado_propriedade""",5570,-0.158044
"""pct_lixo_terreno_encosta_area_…",5570,-0.10193


## Conclusões

Na coorte restaurada, as cinco correlações de saneamento em 2022 permanecem fracas e com os mesmos sinais: água r = 0,1621 e esgoto r = 0,1222; os outros três indicadores têm correlações negativas de baixa magnitude. A amostra reúne 5.570 municípios e não demonstra efeito causal do saneamento. A análise espacial identifica autocorrelação global positiva, mas nenhum cluster local significativo após FDR-BY a 5%. A série histórica deve usar a classificação final corrigida, não os antigos resultados baseados em `DENGUE`.